In [0]:
import re
from collections import defaultdict

In [0]:
#Define input folder and find files
INPUT_DIR = "/Volumes/agrcnaanalytics_workbench/discovery_amendedlostrun/raw_data/Triage_Input/"
files = [
   f for f in dbutils.fs.ls(INPUT_DIR)
   if f.size > 0 and not f.name.startswith("_")
]

In [0]:
#Define regex patterns for normalizing/identifying project files
wc_pattern   = re.compile(r"(^|[ _-])wc([ _-])?loss([ _-])?run([ _-]|\.|$)", re.IGNORECASE)
liab_pattern = re.compile(r"(^|[ _-])liab([ _-])?loss([ _-])?run([ _-]|\.|$)", re.IGNORECASE)
ttd_pattern  = re.compile(r"(^|[ _-])ttd([ _-]|\.|$)", re.IGNORECASE)
ptr_pattern  = re.compile(r"(^|[ _-])ptr([ _-]|\.|$)", re.IGNORECASE)

In [0]:
#Function for normalizing and parsing out client names from file names
def normalize_client_name(filename: str) -> str:
   name = filename.lower()
   name = re.sub(r"\.xlsx$", "", name)
   # Remove file-type tokens
   name = re.sub(r"(^|[ _-])wc([ _-])?loss([ _-])?run([ _-]|$)", " ", name)
   name = re.sub(r"(^|[ _-])liab([ _-])?loss([ _-])?run([ _-]|$)", " ", name)
   name = re.sub(r"(^|[ _-])ttd([ _-]|$)", " ", name)
   name = re.sub(r"(^|[ _-])ptr([ _-]|$)", " ", name)
   # Remove punctuation/spaces
   name = re.sub(r"[^a-z0-9]", "", name)
   return name.strip()

In [0]:
#Uses defined patterns to identify files and assign them to work items for processing
groups = defaultdict(dict)
for f in files:
   name = f.name
   if wc_pattern.search(name):
       key = normalize_client_name(name)
       if key:
           groups[key]["wc_loss_run_path"] = f.path
   elif liab_pattern.search(name):
       key = normalize_client_name(name)
       if key:
           groups[key]["liab_loss_run_path"] = f.path
   elif ttd_pattern.search(name):
       key = normalize_client_name(name)
       if key:
           groups[key]["ttd_path"] = f.path
   elif ptr_pattern.search(name):
       key = normalize_client_name(name)
       if key:
           groups[key]["ptr_path"] = f.path

In [0]:
#Groups work items/file paths for each client. Minimum requirement is a WC Loss Run and/or a LIAB Loss Run (both allowed).
#TTD and PTR are optional; if not provided as separate files they are searched for as sheets inside the loss run workbook(s).
work_items = []
for key, d in groups.items():
   if "wc_loss_run_path" in d or "liab_loss_run_path" in d:
       work_items.append({
           "client_key": key,
           "wc_loss_run_path": d.get("wc_loss_run_path", ""),
           "ttd_path": d.get("ttd_path", ""),
           "ptr_path": d.get("ptr_path", ""),
           "liab_loss_run_path": d.get("liab_loss_run_path", "")
       })
   else:
       print(f"Skipping client '{key}': no WC_Loss_Run or LIAB_Loss_Run file found (TTD/PTR alone is not enough).")
work_items = sorted(work_items, key=lambda x: x["client_key"])
print(f"Found {len(work_items)} client work item(s).")
dbutils.jobs.taskValues.set(key="items", value=work_items)